# Roomify — Colab Launcher

Run cells **top to bottom** on a fresh session.  After Cell 6 prints a `trycloudflare.com` URL, open it in your laptop browser.

| Cell | Purpose | When to run |
|------|---------|-------------|
| 1 | Mount Google Drive, create folder layout | Every session |
| 2 | Clone repo + install deps | First run per Drive |
| 2b | Unzip SUN RGB-D + build subset + manifest | **Once only** |
| 3 | Set HF\_HOME to Drive-backed cache | Every session |
| 4 | Verify GPU — **read output now** | Every session |
| 5 | Load SD 1.5 + smoke-test (skips if already done) | Every session |
| 6 | Start Streamlit + Cloudflare tunnel — **click the URL** | Every session |
| 7 | Reconnect helper (after disconnect, not fresh session) | On reconnect only |
| 8 | Core experiment sweep — 45 images, ~15–30 min on T4 | Once (outputs persist on Drive) |
| 8b | Controlled vs uncontrolled comparison sweep — 90 images | Once (outputs persist on Drive) |
| 9 | Controlled vs uncontrolled pair — single spec verification | Once |
| 10 | VRAM headroom check — run after any generation cell | After generation |
| 11 | AnimateDiff — generate GIFs for top 3 CLIP-scored specs | Once (Phase 8 bonus) |

Support for third party widgets (widgets outside of the ipywidgets package) needs to be enabled separately.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [ ]:
from google.colab import output
output.disable_custom_widget_manager()

In [ ]:
# ── Cell 1: Mount Google Drive and create folder structure ─────────────────
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/roomify')
for subdir in ['data', 'outputs', 'hf_cache']:
    (DRIVE_ROOT / subdir).mkdir(parents=True, exist_ok=True)

print('Drive mounted.  Folder layout:')
for p in sorted(DRIVE_ROOT.iterdir()):
    print(' ', p)

In [ ]:
# ── Cell 2: Clone repo and install Python dependencies ─────────────────────
from pathlib import Path

REPO_URL = 'https://github.com/ben-blake/roomify.git'
REPO_DIR = Path('/content/roomify')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists — pulling latest changes.')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

# Symlink Drive data/outputs into the repo so paths resolve correctly.
DRIVE_ROOT = Path('/content/drive/MyDrive/roomify')
for link_name in ['data', 'outputs']:
    link_path = REPO_DIR / link_name
    target = DRIVE_ROOT / link_name
    if not link_path.exists():
        link_path.symlink_to(target)
        print(f'Symlinked {link_path} -> {target}')

!pip install -q -r requirements.txt
print('\nInstallation complete.')

In [ ]:
# ── Cell 2b: One-time SUN RGB-D setup (skip if manifest.csv already exists) ─
#
# Before running this cell:
#   1. Download SUNRGBD V1 from https://rgbd.cs.princeton.edu/
#   2. Upload the zip to Google Drive at:
#      MyDrive/roomify/data/SUNRGBD.zip
#
# This cell is safe to re-run — it skips any step that is already done.

import zipfile, subprocess, sys
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/roomify/data')
ZIP_PATH   = DRIVE_DATA / 'SUNRGBD.zip'
RAW_DIR    = DRIVE_DATA / 'SUNRGBD'
SUBSET_DIR = DRIVE_DATA / 'sunrgbd_subset'
MANIFEST   = SUBSET_DIR / 'manifest.csv'
REPO_DIR   = Path('/content/roomify')

# ── Step 1: Unzip ──────────────────────────────────────────────────────────
if RAW_DIR.exists():
    print(f'Raw dataset already present at {RAW_DIR} — skipping unzip.')
elif not ZIP_PATH.exists():
    print(f'ERROR: {ZIP_PATH} not found.')
    print('Upload SUNRGBD.zip to MyDrive/roomify/data/ first, then re-run this cell.')
else:
    print(f'Unzipping {ZIP_PATH} -> {RAW_DIR} ...')
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(RAW_DIR)
    print('Unzip complete.')

# ── Step 2: Build subset + manifest ────────────────────────────────────────
if MANIFEST.exists():
    import pandas as pd
    n = len(pd.read_csv(MANIFEST))
    print(f'Manifest already exists ({n} records) — skipping buildSubset.')
elif not RAW_DIR.exists():
    print('Skipping buildSubset — raw dataset not ready yet.')
else:
    print('Building subset (~200 samples across 5 scene types)...')
    result = subprocess.run(
        [
            sys.executable,
            str(REPO_DIR / 'scripts' / 'buildSubset.py'),
            '--sunrgbd-root', str(RAW_DIR),
            '--output-dir',   str(SUBSET_DIR),
            '--samples-per-scene', '40',
            '--copy',
        ],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)
    else:
        print('Subset built successfully.')

print('\nData setup status:')
print(f'  Raw dataset : {"present" if RAW_DIR.exists() else "MISSING"}')
print(f'  Manifest    : {"present" if MANIFEST.exists() else "MISSING"}')

In [ ]:
# ── Cell 3: Point HF_HOME at the Drive-backed cache ────────────────────────
import os
from pathlib import Path

HF_CACHE = Path('/content/drive/MyDrive/roomify/hf_cache')
HF_CACHE.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_CACHE)

print(f'HF_HOME={os.environ["HF_HOME"]}')
print(f'Cache size: ', end='')
!du -sh {HF_CACHE} 2>/dev/null || echo '(empty)'

In [ ]:
# ── Cell 4: Verify GPU and log to session-level file ───────────────────────
import subprocess, json, datetime
from pathlib import Path

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
gpu_info = result.stdout.strip() if result.returncode == 0 else 'GPU not found'
print('GPU:', gpu_info)

log = {
    'timestamp': datetime.datetime.utcnow().isoformat(),
    'gpu': gpu_info,
}
log_path = Path('/content/roomify/session_gpu.json')
log_path.write_text(json.dumps(log, indent=2))
print('Logged to', log_path)

In [ ]:
# ── Cell 5: Load SD 1.5 + smoke-test ───────────────────────────────────────
# Skips the full pipeline load (and generation) if smoke_test.png already exists
# so reconnect sessions don't waste VRAM before Streamlit starts.
# If a fresh load is needed, VRAM is freed after generation so the Streamlit
# subprocess (Cell 6) gets a clean slate.
import sys, os
from pathlib import Path
from IPython.display import display

sys.path.insert(0, '/content/roomify/src')

HF_CACHE   = Path(os.environ.get('HF_HOME', '/content/drive/MyDrive/roomify/hf_cache'))
MODEL_ID   = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
cache_dir  = HF_CACHE / 'hub' / 'models--stable-diffusion-v1-5--stable-diffusion-v1-5'
weights_cached = cache_dir.exists() and any(cache_dir.rglob('*.safetensors'))

out_path = Path('/content/roomify/outputs/smoke_test.png')
out_path.parent.mkdir(parents=True, exist_ok=True)

if out_path.exists():
    print(f'Smoke test already complete ({out_path}) — skipping model load and generation.')
    print('Notebook-process VRAM is free for Streamlit.')
    display(__import__('PIL').Image.open(out_path))
else:
    if weights_cached:
        print(f'Weights found in cache ({cache_dir}) — loading from Drive (fast).')
    else:
        print('Weights not cached — downloading (~4 GB, first run only)...')

    import torch
    from diffusers import StableDiffusionPipeline

    pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        safety_checker=None,
    )
    pipe = pipe.to('cuda')
    pipe.enable_attention_slicing()
    print('Pipeline loaded.')

    print('Generating smoke test image...')
    generator = torch.Generator(device='cuda').manual_seed(42)
    image = pipe(
        'A scandinavian bedroom, natural light, 4k interior design photography',
        negative_prompt='low quality, blurry, distorted proportions',
        num_inference_steps=20,
        generator=generator,
    ).images[0]
    image.save(out_path)
    print(f'Smoke test image saved to {out_path}')
    display(image)

    # Free notebook-process VRAM now so the Streamlit subprocess (Cell 6)
    # has a clean slate when it loads the pipeline via getPipeline().
    del pipe
    torch.cuda.empty_cache()
    print('Notebook-process VRAM freed — ready for Cell 6.')

In [ ]:
# ── Cell 6: Start Streamlit + Cloudflare quick tunnel ──────────────────────
# Safe to re-run — kills any existing processes before starting fresh.
# Polls port 8501 until Streamlit is actually accepting connections before
# starting the tunnel, so the URL is live the moment it is printed.
import socket, subprocess, time, re
from pathlib import Path

REPO_DIR = Path("/content/roomify")
PORT     = 8501
CF_BIN   = Path("/usr/local/bin/cloudflared")
CF_LOG   = Path("/content/cloudflared.log")
ST_LOG   = Path("/content/streamlit.log")

# -- Kill any leftover processes from a previous run --
!pkill -f streamlit  2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

# -- Install cloudflared binary if missing --
if not CF_BIN.exists():
    print("Downloading cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CF_BIN}
    !chmod +x {CF_BIN}
    print("cloudflared installed.")

# -- Launch Streamlit in background --
st_proc = subprocess.Popen(
    ["python", "-m", "streamlit", "run", str(REPO_DIR / "app.py"),
     "--server.port", str(PORT), "--server.headless", "true"],
    cwd=str(REPO_DIR),
    stdout=ST_LOG.open("w"),
    stderr=subprocess.STDOUT,
)
print("Streamlit PID", st_proc.pid, "starting on port", PORT, "...")

# -- Wait until port 8501 is actually accepting connections (max 3 min) --
print("Waiting for Streamlit to be ready", end="", flush=True)
ready = False
for _ in range(90):
    if st_proc.poll() is not None:
        print(" CRASHED")
        print("Streamlit log:")
        !tail -30 {ST_LOG}
        raise SystemExit("Streamlit crashed. Fix errors above before continuing.")
    try:
        s = socket.create_connection(("localhost", PORT), timeout=1)
        s.close()
        ready = True
        break
    except OSError:
        print(".", end="", flush=True)
        time.sleep(2)

if not ready:
    print(" TIMEOUT")
    print("Streamlit log:")
    !tail -30 {ST_LOG}
    raise SystemExit("Streamlit did not start within 3 minutes.")

print(" ready!")

# -- Launch Cloudflare tunnel now that Streamlit is confirmed live --
cf_log_fh = CF_LOG.open("w")
cf_proc = subprocess.Popen(
    [str(CF_BIN), "tunnel", "--url", "http://localhost:" + str(PORT)],
    stdout=cf_log_fh,
    stderr=cf_log_fh,
)

# -- Poll the log file for the public URL (max 30 s) --
tunnel_url = None
for _ in range(30):
    time.sleep(1)
    content = CF_LOG.read_text()
    match = re.search(r"https://[\w-]+\.trycloudflare\.com", content)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print()
    print("=== Roomify is live at:", tunnel_url, "===")
else:
    print("WARNING: Tunnel URL not found after 30 s.")
    print("cloudflared log:")
    !tail -20 {CF_LOG}


In [ ]:
# ── Cell 7: Reconnect helper ────────────────────────────────────────────────
# Run ONLY after a Colab disconnect — not on a fresh session.
import os, subprocess, time, re
from pathlib import Path
from google.colab import drive

print('Re-mounting Google Drive...')
drive.mount('/content/drive', force_remount=True)

os.environ['HF_HOME'] = '/content/drive/MyDrive/roomify/hf_cache'

!pkill -f streamlit  2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

REPO_DIR = Path('/content/roomify')
PORT     = 8501
CF_BIN   = Path('/usr/local/bin/cloudflared')
CF_LOG   = Path('/content/cloudflared.log')
ST_LOG   = Path('/content/streamlit.log')

%cd {REPO_DIR}

# -- Launch Streamlit --
st_proc = subprocess.Popen(
    ['python', '-m', 'streamlit', 'run', str(REPO_DIR / 'app.py'),
     '--server.port', str(PORT), '--server.headless', 'true'],
    cwd=str(REPO_DIR),
    stdout=ST_LOG.open('w'),
    stderr=subprocess.STDOUT,
)
print(f'Streamlit PID {st_proc.pid} starting...')
time.sleep(5)

if st_proc.poll() is not None:
    print('ERROR: Streamlit exited. Log:')
    !tail -20 {ST_LOG}
else:
    print('Streamlit running.')

# -- Launch Cloudflare tunnel (log to file — avoids pipe-buffer deadlock) --
cf_log_fh = CF_LOG.open('w')
cf_proc = subprocess.Popen(
    [str(CF_BIN), 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=cf_log_fh,
    stderr=cf_log_fh,
)

tunnel_url = None
for _ in range(30):
    time.sleep(1)
    content = CF_LOG.read_text()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', content)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print(f'\n=== Roomify reconnected at: {tunnel_url} ===')
else:
    print('WARNING: Tunnel URL not found after 30 s.')
    print('cloudflared log:')
    !tail -20 {CF_LOG}

In [ ]:
# ── Cell 8: Core experiment sweep (45 images) ──────────────────────────────
# Runs 5 specs × 3 strategies × 3 seeds (uncontrolled).
# Fulfils Phase 3 deferred: one sample per scene type with descriptive strategy.
# Estimated time: ~5–10 min on A100, ~15–30 min on T4.
#
# Outputs land at: /content/drive/MyDrive/roomify/outputs/<runId>/

import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/content/roomify')
env = {**os.environ, 'PYTHONPATH': str(REPO_DIR / 'src')}

result = subprocess.run(
    [sys.executable, '-m', 'roomify.cli', 'sweep',
     '--config', str(REPO_DIR / 'configs/experiments/core.yaml')],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)
if result.stderr:
    print('--- stderr ---')
    print(result.stderr)

if result.returncode != 0:
    print(f'\nERROR: sweep exited with code {result.returncode}')
else:
    print('\nSweep complete.  Outputs are on Drive under outputs/.')

In [ ]:
# ── Cell 8b: Controlled vs uncontrolled comparison sweep (90 images) ─────────────
# Runs 5 specs × 3 strategies × 2 (ctrl/unctrl) × 3 seeds = 90 images.
# Produces the ctrl+unctrl pairs needed for 03_evaluation.ipynb Cell 7.
# Estimated time: ~10–20 min on A100, ~30–60 min on T4.
#
# Outputs land at: /content/drive/MyDrive/roomify/outputs/<timestamp>_core_comparison/

import os, sys, subprocess
from pathlib import Path

REPO_DIR = Path("/content/roomify")
env = {**os.environ, "PYTHONPATH": str(REPO_DIR / "src")}

result = subprocess.run(
    [sys.executable, "-m", "roomify.cli", "sweep",
     "--config", str(REPO_DIR / "configs" / "experiments" / "core_comparison.yaml")],
    env=env, capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print("--- stderr ---")
    print(result.stderr)

In [ ]:
# ── Cell 9: Controlled vs uncontrolled pair (Phase 4 deferred) ─────────────
# Picks the first bedroom record from the manifest, then generates the same
# spec + seed twice: once plain (uncontrolled) and once depth-conditioned.
# Eyeball the two images to verify ControlNet follows the reference layout.
#
# Requires Cell 2b to have been run (manifest must exist on Drive).

import os, subprocess, sys
import pandas as pd
from pathlib import Path

REPO_DIR      = Path('/content/roomify')
MANIFEST_PATH = Path('/content/drive/MyDrive/roomify/data/sunrgbd_subset/manifest.csv')
env = {**os.environ, 'PYTHONPATH': str(REPO_DIR / 'src')}

if not MANIFEST_PATH.exists():
    print('ERROR: manifest.csv not found. Run Cell 2b first.')
else:
    df = pd.read_csv(MANIFEST_PATH)
    bedroom_rows = df[df['sceneType'] == 'bedroom']
    ref_id = bedroom_rows.iloc[0]['id'] if not bedroom_rows.empty else df.iloc[0]['id']
    print(f'Reference image ID: {ref_id}\n')

    for label, extra_args in [
        ('uncontrolled', []),
        ('controlled (depth)', ['--control', 'depth', '--ref-image', ref_id]),
    ]:
        print(f'=== Generating {label} ===')
        result = subprocess.run(
            [
                sys.executable, '-m', 'roomify.cli', 'generate',
                '--spec',     str(REPO_DIR / 'configs/examples/bedroom_01.yaml'),
                '--strategy', 'descriptive',
                '--seed',     '42',
            ] + extra_args,
            cwd=str(REPO_DIR),
            capture_output=True,
            text=True,
            env=env,
        )
        print(result.stdout)
        if result.stderr:
            print('--- stderr ---')
            print(result.stderr)
        if result.returncode != 0:
            print(f'ERROR: exited with code {result.returncode}')
        print()

    print('Done. Check outputs/ on Drive — compare the two bedroom_01 runs side by side.')

In [ ]:
# ── Cell 10: VRAM headroom check ───────────────────────────────────────────
# Run after Cell 8 (and optionally Cell 9) to confirm VRAM usage is comfortable.
# If memory.used / memory.total is above ~85%, enable CPU offload in pipeline.py.
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv,noheader

In [ ]:
# ── Cell 11: AnimateDiff — top 3 CLIP-scored specs ──────────────────────────
# Calls the Python API directly (no subprocess) so HF download progress bars
# and generation output stream live. First run downloads the MotionAdapter
# (~300 MB) into the Drive-backed HF cache — expect ~5-10 min on first run.
#
# Top 3 CLIP scores (from examples/phase7/METRICS.md):
#   1. kitchen_01  + styleAnchored  seed=123  CLIP=0.3529
#   2. bathroom_01 + descriptive    seed=7    CLIP=0.3500
#   3. kitchen_01  + descriptive    seed=123  CLIP=0.3448

import dataclasses, json as _json, subprocess, time
from datetime import datetime, timezone
from pathlib import Path
import sys

REPO_DIR = Path("/content/roomify")
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from roomify.animateDiff import AnimateDiffGenerator, framesToGif, MOTION_ADAPTER_ID, SD_MODEL_ID
from roomify.paths import getOutputDir
from roomify.promptBuilder import RoomSpec, buildPrompt

RUNS = [
    ("kitchen_01",  "styleAnchored", 123),
    ("bathroom_01", "descriptive",   7),
    ("kitchen_01",  "descriptive",   123),
]

SPECS = {
    "kitchen_01": RoomSpec(
        id="kitchen_01", roomType="kitchen", size="10x12 ft", style="industrial",
        furniture=["island", "barstools", "open shelving", "range hood"],
        lighting="pendant lights over island", mood="clean, functional",
    ),
    "bathroom_01": RoomSpec(
        id="bathroom_01", roomType="bathroom", size="6x8 ft", style="boho",
        furniture=["freestanding tub", "vanity", "woven baskets", "plants"],
        lighting="soft sconce lighting", mood="relaxing, organic",
    ),
}

print("Loading AnimateDiff pipeline (downloads MotionAdapter on first run)...")
gen = AnimateDiffGenerator()
gen.load()  # progress bars stream here
print("Pipeline ready.")

try:
    git_sha = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
        text=True, stderr=subprocess.DEVNULL,
    ).strip()
except Exception:
    git_sha = "unknown"

for spec_id, strategy, seed in RUNS:
    room_spec = SPECS[spec_id]
    positive, negative = buildPrompt(room_spec, strategy)
    label = spec_id + "_" + strategy + "_s" + str(seed)
    print()
    print("=== Animating", label, "===")
    t0 = time.monotonic()
    frames = gen.generate(positive, negative, seed=seed, steps=25, numFrames=16)
    elapsed = round(time.monotonic() - t0, 2)
    print("Generated", len(frames), "frames in", elapsed, "s")

    ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%S")
    run_id = ts + "_" + label
    out_dir = getOutputDir() / run_id
    out_dir.mkdir(parents=True, exist_ok=True)
    gif_path = out_dir / "anim.gif"
    framesToGif(frames, gif_path, fps=8)
    print("Saved:", gif_path)

    run_json = {
        "runId": run_id, "type": "animate",
        "spec": dataclasses.asdict(room_spec),
        "strategy": strategy, "model": SD_MODEL_ID,
        "motionAdapter": MOTION_ADAPTER_ID,
        "seed": seed, "steps": 25, "guidanceScale": 7.5,
        "numFrames": 16, "fps": 8,
        "prompt": positive, "negativePrompt": negative,
        "gifPath": str(gif_path), "gitSha": git_sha,
        "timings": {"generateSec": elapsed},
    }
    (out_dir / "run.json").write_text(_json.dumps(run_json, indent=2))

print()
print("Done. GIFs saved to", str(getOutputDir()))
